In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("orvile/gastric-cancer-histopathology-tissue-image-dataset")

print("Path to dataset files:", path)

100%|██████████| 3.03G/3.03G [01:49<00:00, 29.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/orvile/gastric-cancer-histopathology-tissue-image-dataset/versions/1


In [11]:
import tensorflow as tf

# ===== CONFIG =====
DATA_PATH = "/root/.cache/kagglehub/datasets/orvile/gastric-cancer-histopathology-tissue-image-dataset/versions/1/HMU-GC-HE-30K/all_image"
IMG_SIZE = (224,224)
BATCH_SIZE = 32
SEED = 123

# ===== TRAIN (70%) =====
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# ===== TEMP (30%) =====
temp_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# ===== SPLIT TEMP → VAL (15%) + TEST (15%) =====
cardinality = tf.data.experimental.cardinality(temp_ds).numpy()

val_size = int(0.5 * cardinality)

val_ds = temp_ds.take(val_size)
test_ds = temp_ds.skip(val_size)

# ===== CLASS NAMES =====
class_names = train_ds.class_names

# ===== PERFORMANCE BOOST =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

# ===== INFO =====
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train_ds))
print("Val batches:", tf.data.experimental.cardinality(val_ds))
print("Test batches:", tf.data.experimental.cardinality(test_ds))

Found 31096 files belonging to 8 classes.
Using 21768 files for training.
Found 31096 files belonging to 8 classes.
Using 9328 files for validation.
Classes: ['ADI', 'DEB', 'LYM', 'MUC', 'MUS', 'NOR', 'STR', 'TUM']
Train batches: tf.Tensor(681, shape=(), dtype=int64)
Val batches: tf.Tensor(146, shape=(), dtype=int64)
Test batches: tf.Tensor(146, shape=(), dtype=int64)


In [13]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [14]:
base_model = tf.keras.applications.ResNet50(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [15]:
base_model.trainable = True

# Freeze first few layers (keep low-level features)
for layer in base_model.layers[:100]:
    layer.trainable = False

In [16]:
model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255),

    base_model,

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

In [17]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="/content/drive/MyDrive/checkpoints/model_epoch_{epoch:02d}.keras",
    save_freq="epoch",   # save every epoch
    save_best_only=False, # save ALL epochs
    verbose=1
)

In [17]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [18]:
import os
os.makedirs("/content/drive/MyDrive/checkpoints", exist_ok=True)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="/content/drive/MyDrive/checkpoints/model_epoch_{epoch:02d}.keras",
    save_freq="epoch",
    save_best_only=False,
    verbose=1

)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",      # watch validation loss
    patience=3,              # stop if no improvement for 3 epochs
    restore_best_weights=True,
    verbose=1
)


best_model = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

In [19]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[checkpoint, early_stop, best_model]
)

Epoch 1/50
778/778 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step - accuracy: 0.2995 - loss: 1.7991
Epoch 1: saving model to /content/drive/MyDrive/checkpoints/model_epoch_01.keras

Epoch 1: val_accuracy improved from -inf to 0.38013, saving model to /content/drive/MyDrive/best_model.keras
778/778 ━━━━━━━━━━━━━━━━━━━━ 199s 218ms/step - accuracy: 0.2996 - loss: 1.7990 - val_accuracy: 0.3801 - val_loss: 1.5710
Epoch 2/50
778/778 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.3958 - loss: 1.5389
Epoch 2: saving model to /content/drive/MyDrive/checkpoints/model_epoch_02.keras

Epoch 2: val_accuracy improved from 0.38013 to 0.40505, saving model to /content/drive/MyDrive/best_model.keras
778/778 ━━━━━━━━━━━━━━━━━━━━ 151s 194ms/step - accuracy: 0.3958 - loss: 1.5388 - val_accuracy: 0.4050 - val_loss: 1.5511
Epoch 3/50
778/778 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - accuracy: 0.4386 - loss: 1.4438
Epoch 3: saving model to /content/drive/MyDrive/checkpoints/model_epoch_03.keras

Epoch 3: val_accuracy impro

In [20]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "/content/drive/MyDrive/best_model.keras"
)

In [28]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=70,              # increased patience
    restore_best_weights=True,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/checkpoints/model_epoch_{epoch:02d}.keras",
    save_freq="epoch",
    verbose=1
)

In [24]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[checkpoint, early_stop]
)

Epoch 1/100
778/778 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step - accuracy: 0.4920 - loss: 1.3203
Epoch 1: saving model to /content/drive/MyDrive/checkpoints/model_epoch_01.keras
778/778 ━━━━━━━━━━━━━━━━━━━━ 189s 210ms/step - accuracy: 0.4920 - loss: 1.3203 - val_accuracy: 0.4764 - val_loss: 1.3791
Epoch 2/100
778/778 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.5219 - loss: 1.2520
Epoch 2: saving model to /content/drive/MyDrive/checkpoints/model_epoch_02.keras
778/778 ━━━━━━━━━━━━━━━━━━━━ 148s 190ms/step - accuracy: 0.5220 - loss: 1.2519 - val_accuracy: 0.4543 - val_loss: 1.4622
Epoch 3/100
778/778 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - accuracy: 0.5618 - loss: 1.1603
Epoch 3: saving model to /content/drive/MyDrive/checkpoints/model_epoch_03.keras
778/778 ━━━━━━━━━━━━━━━━━━━━ 147s 189ms/step - accuracy: 0.5618 - loss: 1.1603 - val_accuracy: 0.4305 - val_loss: 1.7143
Epoch 4/100
778/778 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - accuracy: 0.5955 - loss: 1.0645
Epoch 4: saving model to /content/driv

In [25]:
model.save("/content/drive/MyDrive/overfit_model_backup.keras")

In [26]:
import tensorflow as tf

base_model = tf.keras.applications.ResNet50(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

# Freeze ALL layers
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(8, activation="softmax")
])

In [27]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [1]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=200,
    callbacks=[checkpoint, early_stop]
)

NameError: name 'model' is not defined

### Prediction

In [22]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "/content/drive/MyDrive/checkpoints/model_epoch_80.keras"
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [23]:
class_names = ['ADI','DEB','LYM','MUC','MUS','NOR','STR','TUM']


In [24]:
results = model.evaluate(test_ds)
print(results)

 29/146 ━━━━━━━━━━━━━━━━━━━━ 11:42 6s/step - accuracy: 0.3500 - loss: 1.5900

KeyboardInterrupt: 

In [27]:
# Get one batch
for images, labels in test_ds.take(1):
    img = images[0]      # first image
    true_label = labels[0]

# Add batch dimension and predict
pred = model.predict(tf.expand_dims(img, 0))

print("Raw prediction:", pred)
print("True label:", class_names[true_label.numpy()])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
Raw prediction: [[8.1574464e-05 9.7865954e-02 8.8094085e-01 1.4647662e-03 2.9922822e-03
  9.7622909e-03 4.9316729e-03 1.9605523e-03]]
True label: LYM


In [ ]:
import os

folder = "/content/test_images/"

for file in os.listdir(folder):
    path = folder + file

    img = load_img(path, target_size=(224,224))
    arr = img_to_array(img)/255.0
    arr = np.expand_dims(arr,0)

    pred = model.predict(arr)

    print(file, class_names[np.argmax(pred)])

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ===== CREATE FOLDERS =====
base_dir = "/content/sample_images"
os.makedirs(base_dir, exist_ok=True)

sets = {
    "train": train_ds,
    "val": val_ds,
    "test": test_ds
}

# ===== SAVE FUNCTION =====
def save_images(dataset, set_name, n=10):
    save_path = f"{base_dir}/{set_name}"
    os.makedirs(save_path, exist_ok=True)

    count = 0

    for images, labels in dataset:
        for i in range(len(images)):
            if count >= n:
                return

            img = images[i].numpy().astype("uint8")
            label = class_names[labels[i]]

            plt.imsave(
                f"{save_path}/{set_name}_{count}_{label}.jpg",
                img
            )

            count += 1

# ===== SAVE 10 FROM EACH =====
for name, ds in sets.items():   
    save_images(ds, name, 10)

print("Done! Images saved in:", base_dir)

Done! Images saved in: /content/sample_images
